<a href="https://colab.research.google.com/github/dcthyun0308/ESAA/blob/main/ESAA_OB_week05_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 신용카드 고객 세그먼트 분류 AI 경진대회

1) 코드 흐름 요약
- 데이터 전처리 및 병합 : 제공된 전력 수요, 태양광 발전량 데이터와 기상청 예보 데이터를 시간축을 기준으로 병합, 3시간 단위의 기상 예보 데이터를 시계열 보간을 통해 1시간 단위로 세분화하여 예측 정밀도를 높임.
- 피처 엔지니어링 : 시간 변수 - 시간(Hour), 요일(Day of week), 월(Month) 등을 Sine/Cosine 인코딩하여 주기성을 반영 / Lag 변수 - 24시간 전, 1주일 전의 수요 및 발전량을 파생 변수로 생성하여 시계열적 연속성을 학습 / 기상 변수: 체감 온도, 불쾌지수, 일사량의 이동 평균 등을 생성하여 날씨의 영향을 극대화
- 모델링 : LightGBM과 CatBoost, 앙상블
- 검증 및 예측 : TimeSeriesSplit을 사용하여 미래 시점의 데이터를 예측하는 시계열 데이터 특성에 맞는 교차 검증을 수행함.

2) 새롭게 알게 된 내용 / 배울 점
- 태양광 예측의 특수성: 단순히 기온뿐만 아니라 '운량'과 '일사량'의 상관관계를 모델이 잘 파악하도록 기상 변수 간의 상호작용 피처를 생성한 점이 인상적.

- Cyclic Encoding: 시간 데이터를 단순히 숫자로 넣지 않고 np.sin과 np.cos를 사용하여 23시와 0시가 가깝다는 사실을 모델에 인지시키는 방식이 효과적.

- Post-processing: 발전량이 음수가 나올 수 없는 도메인 지식을 반영하여, 모델 출력값 중 0 이하의 값을 0으로 후처리하는 디테일이 돋보임.

3) 어려운 내용
- 기상 데이터의 보간: 3시간 단위 예보 데이터를 1시간 단위로 선형 또는 스플라인 보간할 때, 실제 기상 변화의 비선형성을 완벽히 반영하기 어렵다는 점이 가장 까다로운 부분임.

4) 주요 코드
- # 1. 시간 변수의 주기성 반영 (Cyclic Encoding)
train['hour_sin'] = np.sin(2 * np.pi * train['hour'] / 24)
train['hour_cos'] = np.cos(2 * np.pi * train['hour'] / 24)

# 2. 기상 변수 보간 및 파생 변수 생성 (체감 온도)
train['perceived_temp'] = 13.12 + 0.6215*train['temp'] - 11.37*(train['wind']**0.16) + 0.3965*(train['wind']**0.16)*train['temp']

# 3. Lag 피처 생성 (24시간 전 발전량/수요량)
for i in [24, 168]: # 1일 전, 1주일 전
    train[f'lag_{i}'] = train.groupby('id')['target'].shift(i)

# 4. LightGBM 모델 정의 및 학습 (하이퍼파라미터 튜닝 포함)
model = LGBMRegressor(n_estimators=1000, learning_rate=0.05, colsample_bytree=0.8)
model.fit(X_train, y_train, eval_set=[(X_val, y_val)], early_stopping_rounds=50, verbose=100)